# Fiche 2 — Filtrer, mesurer le remplissage, agréger avec `groupby` et `pivot_table`

| | |
|---|---|
| 🎯 **Objectif** | Passer de 886 930 lignes à un tableau pays × indicateurs exploitable |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 1 (maille, format large / long) |
| 🔗 **Dans le projet** | Exercice partie 2 |

> **L'idée clé.** On réduit le périmètre pour une seule raison : rendre l'analyse **faisable par un humain**
> sans perdre ce qui compte pour la question métier. Chaque filtre doit donc être justifié et **chiffré** (combien
> de lignes il supprime).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from edstats import viz
from edstats.io import load_tables

viz.apply_style()
pd.set_option("display.max_colwidth", 70)

tables = load_tables()
data, series = tables["Data"], tables["Series"]

## 1. Filtrer : masque booléen, `isin`, `~`

Un filtre pandas se fait en deux temps : on construit un **masque** (une colonne de True/False), puis on l'applique.

In [2]:
masque = series["Topic"] == "Tertiary"
print(masque.head(3).tolist(), "→", masque.sum(), "indicateurs « Tertiary »")

familles = ["Tertiary", "Secondary", "Population"]
print(series["Topic"].isin(familles).sum(), "indicateurs dans l'une des 3 familles")
print((~series["Topic"].isin(familles)).sum(), "indicateurs dans aucune des 3 familles")

[False, False, False] → 158 indicateurs « Tertiary »
627 indicateurs dans l'une des 3 familles
3038 indicateurs dans aucune des 3 familles


⚠️ **Piège classique** : pour combiner deux conditions, on écrit `&` (et) et `|` (ou), **avec des parenthèses** :
`(a == 1) & (b > 2)`. Les mots-clés `and` et `or` provoquent une erreur sur une série :

In [3]:
try:
    series[series["Topic"] == "Tertiary" and series["Topic"] == "Secondary"]
except ValueError as erreur:
    print("ValueError :", erreur)

# La bonne écriture (ici avec « ou », car un indicateur n'a qu'une famille)
print(len(series[(series["Topic"] == "Tertiary") | (series["Topic"] == "Secondary")]), "indicateurs")

ValueError : The truth value of a Series is ambiguous. Use a.empty, a.item(), a.any() or a.all().
414 indicateurs


Retiens le message : « *The truth value of a Series is ambiguous* » signifie presque toujours `and` au lieu de `&`.

## 2. Toujours garder une trace : avant / après

In [4]:
from edstats.quality import FilterLog

from edstats.cleaning import find_aggregates  # la règle de la fiche 1, rangée dans une fonction

journal = FilterLog()
faux_pays = find_aggregates(tables["Country"])
data_pays = journal.apply(data, ~data["Country Code"].isin(faux_pays), "retrait des faux pays (fiche 1)")
codes_tertiaire = series.loc[series["Topic"] == "Tertiary", "Series Code"]
data_tertiaire = journal.apply(data_pays, data_pays["Indicator Code"].isin(codes_tertiaire), "familles : Tertiary")
journal.to_frame()

,etape,lignes_avant,lignes_apres,lignes_supprimees
0,retrait des faux pays (fiche 1),886930,795305,91625
1,familles : Tertiary,795305,32116,763189


## 3. Choisir les années avec `np.arange`

Plutôt que de taper `["1970", "1971", …]` à la main (source d'oublis), on **génère** la liste :

In [5]:
annees_a_garder = [str(a) for a in np.arange(2010, 2015)]  # 2015 exclu : arange s'arrête AVANT la borne haute
print(annees_a_garder)

['2010', '2011', '2012', '2013', '2014']


⚠️ **Toutes les colonnes d'années ne se valent pas.** Regarde le remplissage tous les 5 ans jusqu'en 2100 :

In [6]:
taux_rempli = data_pays[[str(a) for a in range(2000, 2101, 5)]].notna().mean().mul(100).round(1)
taux_rempli.to_frame("% renseigné").T

,2000,2005,2010,2015,2020,2025,2030,2035,2040,2045,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
% renseigné,21.2,22.1,29.4,16.3,6.4,6.4,6.4,6.4,6.4,6.4,...,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4,6.4


Après 2017, le jeu de données ne peut pas contenir d'observations : ce sont des **projections** (scénarios de
population du Wittgenstein Centre). On ne les moyenne **jamais** avec des valeurs observées. On peut les utiliser à
part, pour répondre à « comment le potentiel va-t-il évoluer ? ».

## 4. Mesurer le remplissage avec `groupby`

La question « pour chaque indicateur, quelle part des années est renseignée ? » se traduit en pandas par :
*grouper par indicateur, puis compter les valeurs non vides et diviser par le nombre total de lignes*.

- `count()` compte les valeurs **non manquantes** ;
- `size()` compte **toutes** les lignes.

In [7]:
long = data_tertiaire.melt(id_vars=["Country Code", "Indicator Code"], value_vars=annees_a_garder,
                           var_name="annee", value_name="valeur")
groupes = long.groupby("Indicator Code")["valeur"]
# part des couples (pays, année) renseignés, pour chaque indicateur
proportion_renseignee_par_indicateur = (groupes.count() / groupes.size()).sort_values(ascending=False)
proportion_renseignee_par_indicateur.head(8).round(2)

Indicator Code
SE.TER.ENRL          0.62
SE.TER.ENRL.FE       0.61
SE.TER.ENRL.FE.ZS    0.61
SE.TER.ENRR          0.60
SE.TER.ENRR.FE       0.59
SE.TER.ENRR.MA       0.59
SE.ENR.TERT.FM.ZS    0.59
UIS.GOER.56          0.58
Name: valeur, dtype: float64

C'est exactement ce que fait `edstats.quality.fill_rate` : la fonction évite de réécrire ces 2 lignes pour
chaque regroupement (par année, par indicateur, par pays…).

In [8]:
from edstats.quality import fill_rate

fill_rate(long, by="annee", value="valeur").round(3).to_frame("part renseignée").T

annee,2010,2011,2012,2013,2014
part renseignée,0.344,0.357,0.371,0.386,0.335


## 5. Agréger avec `pivot_table`

Objectif : **une ligne = un pays, une colonne = un indicateur**. Comme il y a 5 années par (pays, indicateur), il
faut les **résumer** en un seul nombre : c'est le rôle de `aggfunc`.

In [9]:
codes = ["SE.TER.ENRR", "SE.TER.ENRL"]
extrait = long[long["Indicator Code"].isin(codes)]
pays_indicateurs = extrait.pivot_table(index="Country Code", columns="Indicator Code", values="valeur",
                                       aggfunc="median")
pays_indicateurs.head()

Indicator Code,SE.TER.ENRL,SE.TER.ENRR
Country Code,,
ABW,2330.0,37.357700
AFG,180189.0,6.209390
AGO,180738.5,8.434830
ALB,160839.0,58.565491
AND,498.0,NaN


### Moyenne ou médiane ? La démonstration

Imaginons un pays dont une année contient une erreur de saisie (un zéro en trop) :

In [10]:
serie = pd.Series([40.1, 41.0, 41.8, 420.0, 43.5], index=annees_a_garder, name="taux (%)")
print(serie.to_dict())
print(f"moyenne = {serie.mean():.1f}   médiane = {serie.median():.1f}")

{'2010': 40.1, '2011': 41.0, '2012': 41.8, '2013': 420.0, '2014': 43.5}
moyenne = 117.3   médiane = 41.8


La moyenne vaut **117**, une valeur impossible pour un taux. La médiane (**41,8**) ignore l'année aberrante.
C'est pourquoi le corrigé agrège avec la médiane.

⚠️ **Piège : agréger trop peu de points.** Une « médiane » calculée sur 1 année n'est qu'une valeur isolée. Le
corrigé exige au moins 3 années :

In [11]:
nb_annees = extrait.pivot_table(index="Country Code", columns="Indicator Code", values="valeur", aggfunc="count")
fiable = pays_indicateurs.where(nb_annees >= 3)  # where garde la valeur si la condition est vraie, sinon NaN
print("valeurs avant :", pays_indicateurs.notna().sum().sum(), " après exigence de 3 années :", fiable.notna().sum().sum())

valeurs avant : 333  après exigence de 3 années : 271


### ✍️ À toi de jouer

Calcule, **pour chaque année** de 2010 à 2014, la proportion de pays ayant une valeur pour l'indicateur `IT.NET.USER.P2` (utilisateurs d'internet). Quelle année est la mieux renseignée ?

In [12]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
internet = data_pays[data_pays["Indicator Code"] == "IT.NET.USER.P2"]
internet_long = internet.melt(id_vars="Country Code", value_vars=annees_a_garder,
                              var_name="annee", value_name="valeur")
fill_rate(internet_long, by="annee", value="valeur").sort_values(ascending=False)
```

Si tu pars de `data` au lieu de `data_pays`, les agrégats (« World »…) sont comptés comme des pays : vérifie toujours sur quel DataFrame tu travailles.
</details>

## ✅ À retenir

1. Filtre = masque booléen ; `isin` pour une liste ; `~` pour inverser ; `&`/`|` avec parenthèses.
2. Chaque filtre se **chiffre** (lignes avant / après) : c'est ce qui rend la démarche défendable.
3. Taux de remplissage = `count()` / `size()` après un `groupby`.
4. `pivot_table` change la maille ; `aggfunc="median"` résiste aux valeurs aberrantes.
5. On n'agrège pas 1 ou 2 points : on fixe un minimum, écrit **en paramètre**.